In [1]:
import requests
import os
import json
import time
import pandas as pd
import sys
import os
import numpy as np
from datetime import date

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, ".."))
sys.path.insert(0, project_root)

import src.modify_reps
import src.gen_committees

In [2]:
try: 
    df = pd.read_json(os.path.join(project_root, "src", "generated_outputs", "congressmen.json"))
except Exception as e:
    print("There is an issue with the congressmen.json. Quitting.")
    sys.exit()

try: 
    vote_df = pd.read_json(os.path.join(project_root, "src", "generated_outputs", "vote_avg.json"))
except Exception as e:
    print("There is an issue with the vote_avg.json. Quitting.")
    sys.exit()

In [3]:
df[df['bioguideID']=="W000194"]

,bioguideID,name,partyName,state,url,attribution,imageUrl,chamber,startYear,endYear
738,W000194,"Watkins, Wes",Republican,Oklahoma,https://api.congress.gov/v3/member/W000194?for...,Collection of the U.S. House of Representatives,https://www.congress.gov/img/member/w000194_20...,House of Representatives,1977,1991.0
739,W000194,"Watkins, Wes",Republican,Oklahoma,https://api.congress.gov/v3/member/W000194?for...,Collection of the U.S. House of Representatives,https://www.congress.gov/img/member/w000194_20...,House of Representatives,1997,2003.0


Run all the modify_reps.py steps

In [4]:
print(f"{len(df['bioguideID'].unique())} bioguides but {len(df)} entries")

2688 bioguides but 2978 entries


In [16]:
df['term'] = [
    [a, b] for a, b in zip(df['startYear'], df['endYear'])
]

df_combined = df.groupby(['bioguideID', 'partyName', 'chamber'])['term'].agg(list).reset_index()
print(len(df_combined))

df_combined

2772


,bioguideID,partyName,chamber,term
0,A000009,Republican,House of Representatives,"[[1973, 1981.0]]"
1,A000009,Republican,Senate,"[[1981, 1987.0]]"
2,A000014,Democratic,House of Representatives,"[[1985, 1987.0], [1991, 2011.0]]"
3,A000017,Democratic,House of Representatives,"[[1971, 1973.0]]"
4,A000017,Democratic,Senate,"[[1973, 1979.0]]"
...,...,...,...,...
2767,Z000013,Democratic,Senate,"[[1977, 1989.0]]"
2768,Z000014,Republican,House of Representatives,"[[1983, 1987.0]]"
2769,Z000016,Republican,House of Representatives,"[[1967, 1975.0]]"
2770,Z000017,Republican,House of Representatives,"[[2015, 2023.0]]"


In [5]:
debug = 0

df = src.modify_reps.update_endyear(df)
df = src.modify_reps.add_tenure(df)
df = src.modify_reps.normalize_name(df)
df = src.modify_reps.only_current(df)
df = src.modify_reps.merge_in_voting_records(df, vote_df)
df = src.modify_reps.replace_democratic(df)

#Functions you need to do on merged vote and reps:
df = src.modify_reps.get_voter_rank(df)

comm_dict = src.gen_committees.gen_committees()
df = src.modify_reps.merge_in_comms(df, comm_dict)




#    get_absolute_stats(df, input_json_f)

2026-01-22 01:38:42,702 - gen_committees.py - WARNING - Comcode not found for None, Nonetype returned.
2026-01-22 01:38:42,702 - gen_committees.py - WARNING - Comcode not found for P000197, Nonetype returned.
2026-01-22 01:38:42,703 - gen_committees.py - WARNING - Comcode not found for S001176, Nonetype returned.
2026-01-22 01:38:42,704 - gen_committees.py - WARNING - Comcode not found for J000299, Nonetype returned.
2026-01-22 01:38:42,704 - gen_committees.py - WARNING - Comcode not found for C001101, Nonetype returned.
2026-01-22 01:38:42,705 - gen_committees.py - WARNING - Comcode not found for J000294, Nonetype returned.


Now for some stats, for your reference:

In [6]:
print(comm_dict)

{'B001323': ['Committee on Natural Resources', 'Committee on Transportation and Infrastructure', 'Committee on Science, Space, and Technology', 'Vice Chair: Committee on Natural Resources: Energy and Mineral Resources', 'Committee on Natural Resources: Oversight and Investigations', 'Committee on Transportation and Infrastructure: Aviation', 'Committee on Transportation and Infrastructure: Coast Guard and Maritime Transportation', 'Committee on Transportation and Infrastructure: Railroads, Pipelines, and Hazardous Materials', 'Committee on Science, Space, and Technology: Environment', 'Committee on Science, Space, and Technology: Energy', 'Committee on Science, Space, and Technology: Investigations and Oversight'], 'M001212': ['Committee on Agriculture', 'Committee on the Judiciary', 'Vice Chair: Committee on Agriculture: Forestry and Horticulture', 'Committee on Agriculture: General Farm Commodities, Risk Management, and Credit', 'Committee on Agriculture: Livestock, Dairy, and Poultr

In [7]:

print(f"There are {len(df[df['current_member']=="yes"])} current members")

#df.sort_values(by='tenure_current', ascending=True).head()


There are 537 current members


Now you can start piloting your definitions here.

In [8]:
#want to merge in the term
df

,bioguideID,name,partyName,state,url,attribution,imageUrl,chamber,startYear,endYear,...,absent_percent,neither_percent,with_party_percent,neither_rank,absent_rank,with_party_rank,with_party_percentile,neither_percentile,absent_percentile,committees
0,M001244,Ashley Moody,Republican,Florida,https://api.congress.gov/v3/member/M001244?for...,Official U.S. Senate Photo,https://www.congress.gov/img/member/https://bi...,Senate,2025,2031,...,1,1,99,68,21,93,93,68,21,"[Joint Economic Committee, Special Committee o..."
1,W000812,Ann Wagner,Republican,Missouri,https://api.congress.gov/v3/member/W000812?for...,Image courtesy of the Member,https://www.congress.gov/img/member/695fc654dd...,House of Representatives,2013,2027,...,4,0,93,98,260,175,40,22,59,"[Committee on Financial Services, Permanent Se..."
2,R000619,Michael A. Rulli,Republican,Ohio,https://api.congress.gov/v3/member/R000619?for...,Image courtesy of the Member,https://www.congress.gov/img/member/69401dcc8c...,House of Representatives,2024,2028,...,11,1,86,280,404,49,11,64,92,"[Committee on Education and Workforce, Committ..."
3,J000312,James C. Justice,Republican,West Virginia,https://api.congress.gov/v3/member/J000312?for...,Official U.S. Senate Photo,https://www.congress.gov/img/member/67c86b5e61...,Senate,2025,2031,...,4,0,95,31,77,48,48,31,77,"[Committee on Energy and Natural Resources, Sp..."
4,D000628,Neal P. Dunn,Republican,Florida,https://api.congress.gov/v3/member/D000628?for...,Congressional Pictorial Directory,https://www.congress.gov/img/member/115_rp_fl_...,House of Representatives,2017,2027,...,12,0,87,98,411,64,15,22,94,"[Vice Chair: Committee on Energy and Commerce,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
532,G000546,Sam Graves,Republican,Missouri,https://api.congress.gov/v3/member/G000546?for...,Collection of the U.S. House of Representatives,https://www.congress.gov/img/member/g000546_20...,House of Representatives,2001,2027,...,7,0,92,154,362,148,34,35,83,"[Committee on Armed Services, Chair: Committee..."
533,M001143,Betty McCollum,Democrat,Minnesota,https://api.congress.gov/v3/member/M001143?for...,Congressional Pictorial Directory,https://www.congress.gov/img/member/116_rp_mn_...,House of Representatives,2001,2027,...,1,1,98,180,96,374,86,41,22,"[Committee on Appropriations, Committee on App..."
534,C000537,James E. Clyburn,Democrat,South Carolina,https://api.congress.gov/v3/member/C000537?for...,Image courtesy of the Member,https://www.congress.gov/img/member/c000537_20...,House of Representatives,1993,2027,...,2,1,97,266,194,322,74,61,44,"[Committee on Appropriations, Committee on App..."
535,K000009,Marcy Kaptur,Democrat,Ohio,https://api.congress.gov/v3/member/K000009?for...,Image courtesy of the Member,https://www.congress.gov/img/member/k000009_20...,House of Representatives,1983,2027,...,1,0,86,120,145,53,12,27,33,"[Committee on Appropriations, Committee on the..."


chamber                   partyName  
House of Representatives  Democrat       46
                          Republican     46
Senate                    Democrat       36
                          Independent    24
                          Republican     48
Name: duration, dtype: int64


   max_tenure_H_D  max_tenure_H_R  max_tenure_S_D  max_tenure_S_I  \
0            46.0            46.0            36.0            24.0   

   max_tenure_S_R  
0            48.0  


   count_D  count_I  count_R
0      260        2      275


            count_D  count_I  count_R
bioguideID      260        2      275


                   0
max_tenure_H_D  46.0
max_tenure_H_R  46.0
max_tenure_S_D  36.0
max_tenure_S_I  24.0
max_tenure_S_R  48.0
count_D          260
count_I            2
count_R          275
max_tenure_H_I   NaN


In [9]:
df_current = df[df['current_member']=="yes"]

df_current[df_current['committees'].isna()]

,bioguideID,name,partyName,state,url,attribution,imageUrl,chamber,endYear,startYear,...,Abstained,Both,Neither,with_D,with_R,vote_count,with_party_count,with_party_percent,with_party_rank,committees
